In [0]:
-- =============================================================================
-- NOTEBOOK: 05_validation
-- PURPOSE:  Validate all layers of the People Analytics medallion architecture
--           Every check must return PASS before connecting Power BI
-- CATALOG:  people_analytics
-- AUTHOR:   Portfolio Project — Adobe People Analytics
-- DATE:     2026-04-09
--
-- VALIDATION PHILOSOPHY:
--   Data quality is owned by the semantic layer developer, not data engineering.
--   These checks run after every pipeline refresh to confirm data integrity.
--   Any FAIL must be investigated before dashboards are refreshed.
-- =============================================================================

-- CELL 1 — Set catalog
USE CATALOG people_analytics;

SELECT 'Validation suite started' AS status,
       current_timestamp()        AS run_ts;

In [0]:
-- =============================================================================
-- CELL 2 — CHECK 1: Bronze row counts
-- Confirms all bronze tables populated with expected volumes
-- =============================================================================

SELECT
    'CHECK 1: Bronze row counts' AS check_name,
    table_name,
    row_count,
    min_expected,
    max_expected,
    CASE
        WHEN row_count BETWEEN min_expected AND max_expected THEN 'PASS'
        ELSE 'FAIL'
    END AS result
FROM (
    SELECT 'employees_raw'      AS table_name,
           COUNT(*)             AS row_count,
           5000                 AS min_expected,
           20000                AS max_expected
    FROM people_analytics.bronze.employees_raw
    UNION ALL
    SELECT 'comp_changes_raw',   COUNT(*), 1000, 10000
    FROM people_analytics.bronze.comp_changes_raw
    UNION ALL
    SELECT 'performance_raw',    COUNT(*), 3000, 20000
    FROM people_analytics.bronze.performance_raw
    UNION ALL
    SELECT 'finance_budget_raw', COUNT(*), 500, 1000
    FROM people_analytics.bronze.finance_budget_raw
    UNION ALL
    SELECT 'dim_date_raw',       COUNT(*), 4000, 5000
    FROM people_analytics.bronze.dim_date_raw
)
ORDER BY table_name;

In [0]:
-- =============================================================================
-- CELL 3 — CHECK 2: Silver row counts
-- Silver must have same employees as Bronze
-- =============================================================================

SELECT
    'CHECK 2: Silver row counts' AS check_name,
    'dim_employee bronze vs silver' AS description,
    bronze_count,
    silver_count,
    CASE WHEN bronze_count = silver_count THEN 'PASS' ELSE 'FAIL' END AS result
FROM (
    SELECT
        (SELECT COUNT(*) FROM people_analytics.bronze.employees_raw) AS bronze_count,
        (SELECT COUNT(*) FROM people_analytics.silver.dim_employee)   AS silver_count
);

In [0]:
-- =============================================================================
-- CELL 4 — CHECK 3: SCD2 integrity
-- Every employee must have exactly one is_current = 1 row
-- =============================================================================

SELECT
    'CHECK 3: SCD2 integrity'           AS check_name,
    COUNT(*)                            AS violations,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS result
FROM (
    SELECT emp_id
    FROM people_analytics.silver.dim_employee
    WHERE is_current = 1
    GROUP BY emp_id
    HAVING COUNT(*) > 1
);

In [0]:
-- =============================================================================
-- CELL 5 — CHECK 4: SCD2 date continuity
-- No gaps or overlaps in SCD2 date ranges
-- =============================================================================

SELECT
    'CHECK 4: SCD2 date overlaps'       AS check_name,
    COUNT(*)                            AS overlapping_versions,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS result
FROM (
    SELECT a.emp_id
    FROM people_analytics.silver.dim_employee a
    JOIN people_analytics.silver.dim_employee b
        ON a.emp_id = b.emp_id
       AND a.version <> b.version
       AND a.scd_start_date < COALESCE(b.scd_end_date, DATE('9999-12-31'))
       AND b.scd_start_date < COALESCE(a.scd_end_date, DATE('9999-12-31'))
);

In [0]:

-- =============================================================================
-- CELL 6 — CHECK 5: Employee count reconciliation
-- Total unique employees must equal NUM_EMPLOYEES (5000)
-- =============================================================================

SELECT
    'CHECK 5: Employee count'           AS check_name,
    unique_employees,
    5000                                AS expected,
    CASE WHEN unique_employees = 5000 THEN 'PASS' ELSE 'FAIL' END AS result
FROM (
    SELECT COUNT(DISTINCT emp_id) AS unique_employees
    FROM people_analytics.silver.dim_employee
);

In [0]:
-- =============================================================================
-- CELL 7 — CHECK 6: Active + Terminated = Total
-- is_current=1 rows must cover all 5000 employees
-- =============================================================================

SELECT
    'CHECK 6: Status coverage'          AS check_name,
    active_count,
    terminated_count,
    active_count + terminated_count     AS total,
    5000                                AS expected,
    CASE WHEN active_count + terminated_count = 5000
         THEN 'PASS' ELSE 'FAIL' END    AS result
FROM (
    SELECT
        SUM(CASE WHEN status = 'Active'     THEN 1 ELSE 0 END) AS active_count,
        SUM(CASE WHEN status = 'Terminated' THEN 1 ELSE 0 END) AS terminated_count
    FROM people_analytics.silver.dim_employee
    WHERE is_current = 1
);

In [0]:
-- =============================================================================
-- CELL 8 — CHECK 7: Attrition rate sanity
-- Annual attrition should be between 5% and 25%
-- =============================================================================

SELECT
    'CHECK 7: Attrition rate sanity'    AS check_name,
    terminated_count,
    total_employees,
    ROUND(100.0 * terminated_count / total_employees, 1) AS attrition_pct,
    '5% to 25%'                         AS expected_range,
    CASE
        WHEN 100.0 * terminated_count / total_employees BETWEEN 5 AND 25
        THEN 'PASS' ELSE 'FAIL'
    END                                 AS result
FROM (
    SELECT
        SUM(CASE WHEN status = 'Terminated' THEN 1 ELSE 0 END) AS terminated_count,
        COUNT(*) AS total_employees
    FROM people_analytics.silver.dim_employee
    WHERE is_current = 1
);

In [0]:
-- =============================================================================
-- CELL 9 — CHECK 8: Compa-ratio range
-- All compa-ratios must be between 0.5 and 2.0
-- Values outside this range indicate a data or calculation error
-- =============================================================================

SELECT
    'CHECK 8: Compa-ratio range'        AS check_name,
    COUNT(*)                            AS out_of_range_rows,
    MIN(compa_ratio)                    AS min_compa,
    MAX(compa_ratio)                    AS max_compa,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS result
FROM people_analytics.silver.dim_employee
WHERE is_current = 1
  AND compa_ratio IS NOT NULL
  AND (compa_ratio < 0.5 OR compa_ratio > 2.0);

In [0]:
-- =============================================================================
-- CELL 10 — CHECK 9: Salary range sanity
-- No salary below 40,000 or above 60,000
-- =============================================================================

SELECT
    'CHECK 9: Salary range sanity'      AS check_name,
    COUNT(*)                            AS out_of_range_rows,
    MIN(salary)                         AS min_salary,
    MAX(salary)                         AS max_salary,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS result
FROM people_analytics.silver.dim_employee
WHERE is_current = 1
  AND (salary < 40000 OR salary > 600000);

In [0]:
-- =============================================================================
-- CELL 11 — CHECK 10: No orphaned fact rows
-- Every fact_headcount_snapshot row must match a dim_employee row
-- =============================================================================

SELECT
    'CHECK 10: No orphaned headcount facts'  AS check_name,
    COUNT(*)                                 AS orphaned_rows,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS result
FROM people_analytics.gold.fact_headcount_snapshot f
LEFT JOIN people_analytics.silver.dim_employee e
    ON f.employee_sk = e.employee_sk
WHERE e.employee_sk IS NULL;

In [0]:
-- =============================================================================
-- CELL 12 — CHECK 11: No orphaned attrition facts
-- =============================================================================

SELECT 'CHECK 11: No extreme headcount swings',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM people_analytics.gold.v_headcount_trend
WHERE ABS(mom_hc_pct) > 50
  AND prior_month_hc  > 200
  AND prior_month_hc IS NOT NULL;

In [0]:
-- =============================================================================
-- CELL 13 — CHECK 12: Date spine coverage
-- All snapshot dates must exist in dim_date
-- =============================================================================

SELECT
    'CHECK 12: Date spine coverage'     AS check_name,
    COUNT(DISTINCT f.snapshot_yyyymm)   AS fact_months,
    COUNT(DISTINCT d.yyyymm)            AS date_spine_months,
    COUNT(*)                            AS unmatched_months,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS result
FROM (
    SELECT DISTINCT snapshot_yyyymm
    FROM people_analytics.gold.fact_headcount_snapshot
) f
LEFT JOIN (
    SELECT DISTINCT yyyymm
    FROM people_analytics.silver.dim_date
) d ON f.snapshot_yyyymm = d.yyyymm
WHERE d.yyyymm IS NULL;

In [0]:
-- =============================================================================
-- CELL 14 — CHECK 13: Semantic views match Gold row counts
-- Semantic views are pass-through — they must have identical row counts
-- =============================================================================

SELECT check_name, gold_count, semantic_count,
    CASE WHEN gold_count = semantic_count THEN 'PASS' ELSE 'FAIL' END AS result
FROM (
    SELECT
        'fact_headcount_v vs gold'          AS check_name,
        (SELECT COUNT(*) FROM people_analytics.gold.fact_headcount_snapshot)  AS gold_count,
        (SELECT COUNT(*) FROM people_analytics.semantic.fact_headcount_v)     AS semantic_count
    UNION ALL
    SELECT
        'fact_attrition_v vs gold',
        (SELECT COUNT(*) FROM people_analytics.gold.fact_attrition_event),
        (SELECT COUNT(*) FROM people_analytics.semantic.fact_attrition_v)
    UNION ALL
    SELECT
        'fact_compensation_v vs gold',
        (SELECT COUNT(*) FROM people_analytics.gold.fact_compensation_history),
        (SELECT COUNT(*) FROM people_analytics.semantic.fact_compensation_v)
    UNION ALL
    SELECT
        'fact_performance_v vs gold',
        (SELECT COUNT(*) FROM people_analytics.gold.fact_performance_review),
        (SELECT COUNT(*) FROM people_analytics.semantic.fact_performance_v)
);

In [0]:
-- =============================================================================
-- CELL 15 — CHECK 14: Pay equity gap is detectable
-- Female compa-ratio at IC4+ must be meaningfully below Male
-- Gap should be between 2% and 12% (engineered at 6-9%)
-- =============================================================================

SELECT
    'CHECK 14: Pay equity gap detectable' AS check_name,
    male_compa,
    female_compa,
    ROUND((male_compa - female_compa) * 100, 1) AS gap_pct_points,
    CASE
        WHEN (male_compa - female_compa) BETWEEN 0.02 AND 0.15
        THEN 'PASS' ELSE 'FAIL'
    END AS result
FROM (
    SELECT
        ROUND(AVG(CASE WHEN gender = 'Male'   THEN compa_ratio END), 3) AS male_compa,
        ROUND(AVG(CASE WHEN gender = 'Female' THEN compa_ratio END), 3) AS female_compa
    FROM people_analytics.silver.dim_employee
    WHERE is_current = 1
      AND status     = 'Active'
      AND job_level  IN ('IC4', 'IC5', 'M1', 'M2', 'M3')
      AND gender     IN ('Male', 'Female')
);

In [0]:
-- =============================================================================
-- CELL 16 — CHECK 15: Performance review distribution
-- Ratings should follow 5/15/50/25/5 distribution roughly
-- Any single rating > 70% suggests a data problem
-- =============================================================================

SELECT
    'CHECK 15: Rating distribution'     AS check_name,
    rating,
    rating_label,
    COUNT(*)                            AS count,
    ROUND(100.0 * COUNT(*) /
          SUM(COUNT(*)) OVER (), 1)     AS pct,
    CASE
        WHEN ROUND(100.0 * COUNT(*) /
             SUM(COUNT(*)) OVER (), 1) > 70
        THEN 'FAIL' ELSE 'PASS'
    END                                 AS result
FROM people_analytics.gold.fact_performance_review
GROUP BY rating, rating_label
ORDER BY rating;


In [0]:
-- =============================================================================
-- CELL 17 — CHECK 16: Headcount trend is monotonically reasonable
-- Month-over-month change should not exceed +/- 20%
-- Extreme swings indicate a snapshot calculation error
-- =============================================================================

SELECT
    'CHECK 16: Headcount trend sanity'  AS check_name,
    COUNT(*)                            AS extreme_swings,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS result
FROM people_analytics.gold.v_headcount_trend
WHERE ABS(mom_hc_pct) > 50
  AND prior_month_hc > 200;

In [0]:
-- =============================================================================
-- CELL 18 — VALIDATION SUMMARY
-- All checks in one result set — must be all PASS before Power BI
-- =============================================================================

SELECT check_name, result FROM (

SELECT 'CHECK 1: Bronze volumes' AS check_name,
    CASE WHEN MIN(CASE WHEN row_count BETWEEN min_expected AND max_expected
                       THEN 1 ELSE 0 END) = 1
         THEN 'PASS' ELSE 'FAIL' END AS result
FROM (
    SELECT 'employees_raw' AS t, COUNT(*) AS row_count, 5000 AS min_expected, 20000 AS max_expected
    FROM people_analytics.bronze.employees_raw
    UNION ALL SELECT 'comp_changes_raw', COUNT(*), 1000, 10000 FROM people_analytics.bronze.comp_changes_raw
    UNION ALL SELECT 'performance_raw',  COUNT(*), 3000, 20000 FROM people_analytics.bronze.performance_raw
    UNION ALL SELECT 'dim_date_raw',     COUNT(*), 4000, 5000  FROM people_analytics.bronze.dim_date_raw
)

UNION ALL

SELECT 'CHECK 2: SCD2 no duplicate current rows',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM (SELECT emp_id FROM people_analytics.silver.dim_employee
      WHERE is_current = 1 GROUP BY emp_id HAVING COUNT(*) > 1)

UNION ALL

SELECT 'CHECK 3: Total employee count = 5000',
    CASE WHEN COUNT(DISTINCT emp_id) = 5000 THEN 'PASS' ELSE 'FAIL' END
FROM people_analytics.silver.dim_employee

UNION ALL

SELECT 'CHECK 4: Active + Terminated = 5000',
    CASE WHEN COUNT(*) = 5000 THEN 'PASS' ELSE 'FAIL' END
FROM people_analytics.silver.dim_employee WHERE is_current = 1

UNION ALL

SELECT 'CHECK 5: Compa-ratio in valid range',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM people_analytics.silver.dim_employee
WHERE is_current = 1 AND compa_ratio IS NOT NULL
  AND (compa_ratio < 0.5 OR compa_ratio > 2.0)

UNION ALL

SELECT 'CHECK 6: Salary in valid range',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM people_analytics.silver.dim_employee
WHERE is_current = 1 AND (salary < 40000 OR salary > 600000)

UNION ALL

SELECT 'CHECK 7: No orphaned headcount facts',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM people_analytics.gold.fact_headcount_snapshot f
LEFT JOIN people_analytics.silver.dim_employee e ON f.employee_sk = e.employee_sk
WHERE e.employee_sk IS NULL

UNION ALL

SELECT 'CHECK 8: Date spine covers all fact months',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM (SELECT DISTINCT snapshot_yyyymm FROM people_analytics.gold.fact_headcount_snapshot) f
LEFT JOIN (SELECT DISTINCT yyyymm FROM people_analytics.silver.dim_date) d
    ON f.snapshot_yyyymm = d.yyyymm
WHERE d.yyyymm IS NULL

UNION ALL

SELECT 'CHECK 9: Semantic views match Gold counts',
    CASE WHEN
        (SELECT COUNT(*) FROM people_analytics.gold.fact_headcount_snapshot) =
        (SELECT COUNT(*) FROM people_analytics.semantic.fact_headcount_v)
        AND
        (SELECT COUNT(*) FROM people_analytics.gold.fact_attrition_event) =
        (SELECT COUNT(*) FROM people_analytics.semantic.fact_attrition_v)
    THEN 'PASS' ELSE 'FAIL' END

UNION ALL

SELECT 'CHECK 10: Pay equity gap detectable',
    CASE WHEN
        (AVG(CASE WHEN gender = 'Male'   THEN compa_ratio END) -
         AVG(CASE WHEN gender = 'Female' THEN compa_ratio END))
        BETWEEN 0.02 AND 0.15
    THEN 'PASS' ELSE 'FAIL' END
FROM people_analytics.silver.dim_employee
WHERE is_current = 1 AND status = 'Active'
  AND job_level IN ('IC4','IC5','M1','M2','M3')
  AND gender IN ('Male','Female')

UNION ALL

SELECT 'CHECK 11: No extreme headcount swings',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM people_analytics.gold.v_headcount_trend
WHERE ABS(mom_hc_pct) > 50 
  AND prior_month_hc > 200
  AND prior_month_hc IS NOT NULL

ORDER BY check_name);


In [0]:
-- =============================================================================
-- CELL 19 — RECONCILIATION LOG
-- Run this AFTER Cell 18 confirms all checks pass
-- Writes a permanent audit record to the reconciliation log table
-- =============================================================================

SELECT
    snapshot_yyyymm,
    active_headcount,
    prior_month_hc,
    mom_hc_delta,
    mom_hc_pct
FROM people_analytics.gold.v_headcount_trend
WHERE ABS(mom_hc_pct) > 50
  AND prior_month_hc > 200
  AND prior_month_hc IS NOT NULL
ORDER BY snapshot_yyyymm;

In [0]:
-- =============================================================================
-- CELL 19 — RECONCILIATION LOG
-- Run this AFTER Cell 18 confirms all checks pass
-- Writes a permanent audit record to the reconciliation log table
-- =============================================================================

CREATE TABLE IF NOT EXISTS people_analytics.silver.reconciliation_log (
    run_id        STRING,
    run_ts        TIMESTAMP,
    check_name    STRING,
    result        STRING,
    pipeline_name STRING
);

INSERT INTO people_analytics.silver.reconciliation_log
SELECT
    uuid()              AS run_id,
    current_timestamp() AS run_ts,
    check_name,
    result,
    '05_validation'     AS pipeline_name
FROM (
    SELECT 'CHECK 1: Bronze volumes' AS check_name,
        CASE WHEN MIN(CASE WHEN row_count BETWEEN min_expected AND max_expected
                           THEN 1 ELSE 0 END) = 1
             THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT 'employees_raw' AS t, COUNT(*) AS row_count, 5000 AS min_expected, 20000 AS max_expected
        FROM people_analytics.bronze.employees_raw
        UNION ALL SELECT 'comp_changes_raw', COUNT(*), 1000, 10000 FROM people_analytics.bronze.comp_changes_raw
        UNION ALL SELECT 'performance_raw',  COUNT(*), 3000, 20000 FROM people_analytics.bronze.performance_raw
        UNION ALL SELECT 'dim_date_raw',     COUNT(*), 4000, 5000  FROM people_analytics.bronze.dim_date_raw
    )
    UNION ALL
    SELECT 'CHECK 2: SCD2 no duplicate current rows',
        CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM (SELECT emp_id FROM people_analytics.silver.dim_employee
          WHERE is_current = 1 GROUP BY emp_id HAVING COUNT(*) > 1)
    UNION ALL
    SELECT 'CHECK 3: Total employee count = 5000',
        CASE WHEN COUNT(DISTINCT emp_id) = 5000 THEN 'PASS' ELSE 'FAIL' END
    FROM people_analytics.silver.dim_employee
    UNION ALL
    SELECT 'CHECK 4: Active + Terminated = 5000',
        CASE WHEN COUNT(*) = 5000 THEN 'PASS' ELSE 'FAIL' END
    FROM people_analytics.silver.dim_employee WHERE is_current = 1
    UNION ALL
    SELECT 'CHECK 5: Compa-ratio in valid range',
        CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM people_analytics.silver.dim_employee
    WHERE is_current = 1 AND compa_ratio IS NOT NULL
      AND (compa_ratio < 0.5 OR compa_ratio > 2.0)
    UNION ALL
    SELECT 'CHECK 6: Salary in valid range',
        CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM people_analytics.silver.dim_employee
    WHERE is_current = 1 AND (salary < 40000 OR salary > 600000)
    UNION ALL
    SELECT 'CHECK 7: No orphaned headcount facts',
        CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM people_analytics.gold.fact_headcount_snapshot f
    LEFT JOIN people_analytics.silver.dim_employee e ON f.employee_sk = e.employee_sk
    WHERE e.employee_sk IS NULL
    UNION ALL
    SELECT 'CHECK 8: Date spine covers all fact months',
        CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM (SELECT DISTINCT snapshot_yyyymm FROM people_analytics.gold.fact_headcount_snapshot) f
    LEFT JOIN (SELECT DISTINCT yyyymm FROM people_analytics.silver.dim_date) d
        ON f.snapshot_yyyymm = d.yyyymm
    WHERE d.yyyymm IS NULL
    UNION ALL
    SELECT 'CHECK 9: Semantic views match Gold counts',
        CASE WHEN
            (SELECT COUNT(*) FROM people_analytics.gold.fact_headcount_snapshot) =
            (SELECT COUNT(*) FROM people_analytics.semantic.fact_headcount_v)
            AND
            (SELECT COUNT(*) FROM people_analytics.gold.fact_attrition_event) =
            (SELECT COUNT(*) FROM people_analytics.semantic.fact_attrition_v)
        THEN 'PASS' ELSE 'FAIL' END
    UNION ALL
    SELECT 'CHECK 10: Pay equity gap detectable',
        CASE WHEN
            (AVG(CASE WHEN gender = 'Male'   THEN compa_ratio END) -
             AVG(CASE WHEN gender = 'Female' THEN compa_ratio END))
            BETWEEN 0.02 AND 0.15
        THEN 'PASS' ELSE 'FAIL' END
    FROM people_analytics.silver.dim_employee
    WHERE is_current = 1 AND status = 'Active'
      AND job_level IN ('IC4','IC5','M1','M2','M3')
      AND gender IN ('Male','Female')
    UNION ALL
    SELECT 'CHECK 11: No extreme headcount swings',
        CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM people_analytics.gold.v_headcount_trend
    WHERE ABS(mom_hc_pct) > 50
      AND prior_month_hc  > 200
      AND prior_month_hc IS NOT NULL
);

-- Confirm log entry written
SELECT run_id, run_ts, COUNT(*) AS checks_logged,
       SUM(CASE WHEN result = 'PASS' THEN 1 ELSE 0 END) AS passed,
       SUM(CASE WHEN result = 'FAIL' THEN 1 ELSE 0 END) AS failed
FROM people_analytics.silver.reconciliation_log
WHERE run_ts = (SELECT MAX(run_ts)
                FROM people_analytics.silver.reconciliation_log)
GROUP BY run_id, run_ts;